# Sparkle Movie — Système de Recommandation de Films

---

## Contexte

Une plateforme de streaming souhaite améliorer l'expérience utilisateur en proposant des **recommandations de films personnalisées**. Pour cela, nous exploitons le dataset **MovieLens** qui contient les notes attribuées par des utilisateurs réels à des milliers de films.

## Problématique

> **Comment recommander des films pertinents à un utilisateur, en s'appuyant sur ses préférences passées et sur le comportement d'utilisateurs similaires ?**

## Plan du notebook

| Section | Contenu |
|---------|---------|
| 1. Préparation | Installation, session Spark, téléchargement des données |
| 2. Chargement | Import des CSV, exploration, nettoyage |
| 3. Analyse | Films populaires, genres, tendances |
| 4. Visualisations | Graphiques avec matplotlib/seaborn |

---

## 1. Préparation de l'environnement

### 1.1 Dépendances

Les librairies utilisées dans ce projet :

| Librairie | Rôle |
|-----------|------|
| `pyspark` | Traitement distribué des données |
| `pandas` | Manipulation légère des données |
| `matplotlib` / `seaborn` | Visualisations |

In [ ]:
# Décommenter et exécuter une seule fois si les librairies ne sont pas installées
# !pip install pyspark matplotlib seaborn pandas

### 1.2 Session Spark

Apache Spark tourne sur la JVM (Java). On configure `JAVA_HOME` puis on initialise une **SparkSession** — point d'entrée unique pour toutes les opérations Spark.

In [ ]:
import os
os.environ["JAVA_HOME"] = "C:/Program Files/Microsoft/jdk-21.0.10.7-hotspot"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkleMovie")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version} — session démarrée")

### 1.3 Dataset MovieLens

Nous utilisons la version **Small** du dataset MovieLens :
- `ratings.csv` — ~100 000 notes de **610 utilisateurs** sur **9 742 films**
- `movies.csv` — titre et genres des films

La taille réduite est idéale pour le développement ; les mêmes algorithmes s'appliquent au dataset complet (33M de notes).

In [ ]:
import urllib.request
import zipfile

DATA_DIR = "data"
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if not os.path.exists(DATA_DIR):
    print("Téléchargement du dataset MovieLens Small...")
    urllib.request.urlretrieve(MOVIELENS_URL, "ml.zip")
    with zipfile.ZipFile("ml.zip") as z:
        z.extractall(".")
    os.rename("ml-latest-small", DATA_DIR)
    os.remove("ml.zip")
    print("Terminé.")

print("Fichiers disponibles :")
for f in sorted(os.listdir(DATA_DIR)):
    size_kb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1024
    print(f"  {f:<20} {size_kb:>8.1f} Ko")

### 1.4 Vérification des versions

In [ ]:
import pyspark
import pandas as pd
import matplotlib
import seaborn

libs = {
    "PySpark":     pyspark.__version__,
    "Pandas":      pd.__version__,
    "Matplotlib":  matplotlib.__version__,
    "Seaborn":     seaborn.__version__,
}

for lib, version in libs.items():
    print(f"  {lib:<12} {version}")

print("\nEnvironnement pret.")